# Homework: Build a StationXML Inventory for recent KSC deployment

Seven USF stations were deployed at Kennedy Space Center between January 7-12, 2026.
* Each features a Nanometrics Centaur digitizer, and a Nanometrics Trillium Compact Post-Hole 120-s seismometer.
* This is the same equipment we setup in our class a few weeks ago - except we also had infrasound.

Three additional seismic stations were setup by Marshall Space Flight Center, using Guralp and Silicon Audio equipment. You can ignore those for this exercise - or potentially earn some bonus points if you find responses for those too!

## Goal
Create a valid **StationXML** file for this small network using:
- Station coordinates stored in an **Excel** file
- Instrument responses pulled from the **IRIS Nominal Response Library (NRL)**

You will:
1. Load the station list with **pandas** into a pandas DataFrame
2. Create a Response object for a USF Nanometrics station
3. Build an ObsPy **Inventory**
4. Write out a **StationXML** file

> We're keeping it simple: one network, three channels per station (Z, N, E), same equipment at each site.

**Warning:** only attempt this homework after you have completed this week's reading exercises (notebooks 73 and 74)

## Provided file
Your station list is in:

- `Summary_Seismic_Station_List.xlsx`

One column contains photos — **ignore it**.

An alternative version of the Excel file is also provided as a Comma-Separated-Variable (CSV) text file, just in case you cannot read the Excel file. This is easier to read.

(If you open any Excel file with Excel, you can save it as a CSV).

## Make sure you have openpyxl installed in your conda environment!
This is required to read Excel files. CSV files will work without it.

In [ ]:
import sys
!conda install -y openpyxl --prefix {sys.prefix}

## Minimal pandas intro (what you need today)

**pandas** is a Python library for working with tables (“dataframes”).  
For this homework you only need to:
- read an Excel file
- select a few columns
- loop over rows

**Note**: you may need to install openpyxl in conda, and then restart this notebook/kernel. At the command line:

```
conda activate compsci # or whatever your conda env is
conda install openpyxml
```

In [ ]:
import pandas as pd

xlsx_path = r"Summary_Seismic_Station_List.xlsx"
df = pd.read_excel(xlsx_path)

df.head(10)

If for any reason you cannot load the Excel file, you can try the CSV file instead:
```
df = pd.read_csv("Summary_Seismic_Station_List.csv")
```

In [ ]:
df.columns

## Step 1 — Keep only the columns we need

For StationXML, we need at minimum:
- station code (name)
- latitude
- longitude

We will also keep:
- seismometer model (if present)
- digitizer model (if present)

In [ ]:
use_cols = ["Site Name", "lat", "lon", "Seismometer", "Digitizer"]
stations = df[use_cols].copy()

# Ignore the Photo column (already excluded)
stations

## Step 2 — Create a Response object for a USF Nanometrics station

In [ ]:
# Model answer: create a Response object for the USF Nanometrics stations

from obspy.clients.nrl import NRL

nrl = NRL()

FS = 500 # added after reading ahead to Step 3

# USF equipment used at the seven USF stations
sensor_keys = [
    "Nanometrics",
    "Trillium Compact 120 (Vault, Posthole, OBS)",
    "754 V/m/s",
]

datalogger_keys = [
    "Nanometrics",
    "Centaur",
    "40 Vpp (1)",
    "Off",
    "Linear phase",
    str(FS)
]

resp_usf = nrl.get_response(
    sensor_keys=sensor_keys,
    datalogger_keys=datalogger_keys,
)

resp_usf


## Step 3 - Create an ObsPy Inventory

The Inventory object should contain a list of 1 Network object.

The Network object should have a network code, description, and start_date, and contain a list of 7 Station objects.

Each Station object should have a station code, description, and coordinates, and a list of 3 Channel objects.

Each Channel object should have a channel code, a location code, coordinates (StationXML repeats them), depth, and sampling rate, and a Response object.


In [ ]:
# Model answer: build an ObsPy Inventory for the seven USF stations

from obspy.core.inventory import Inventory, Network, Station, Channel, Site
from obspy import UTCDateTime

# -----------------------------
# Network-level metadata
# -----------------------------
NETWORK_CODE = "1R"
NETWORK_DESCRIPTION = "KSC Seismic Network"
START_DATE = UTCDateTime(2026, 1, 7)

# -----------------------------
# Channel metadata
# -----------------------------
CHANNEL_CODES = ["DHZ", "DHN", "DHE"]
LOC_CODE = ""
ELEV_M = 0.0
DEPTH_M = 0.75

def is_usf_station(row):
    seismometer = str(row["Seismometer"]) if not pd.isna(row["Seismometer"]) else ""
    digitizer = str(row["Digitizer"]) if not pd.isna(row["Digitizer"]) else ""

    return ("Trillium" in seismometer) or ("Centaur" in digitizer)

stations_usf = stations[stations.apply(is_usf_station, axis=1)].copy()
stations_usf


## Step 4 — Build Inventory, plot, and write StationXML

In [ ]:
# Model answer: create stations/channels, build inventory, plot, and write StationXML

stations_list = []

for _, row in stations_usf.iterrows():
    sta_code = row["Site Name"]
    sta_lat = float(row["lat"])
    sta_lon = float(row["lon"])

    channels = []
    for chan_code in CHANNEL_CODES:
        if chan_code.endswith("Z"):
            azimuth = 0.0
            dip = -90.0
        elif chan_code.endswith("N"):
            azimuth = 0.0
            dip = 0.0
        else:  # E
            azimuth = 90.0
            dip = 0.0

        ch = Channel(
            code=chan_code,
            location_code=LOC_CODE,
            latitude=sta_lat,
            longitude=sta_lon,
            elevation=ELEV_M,
            depth=DEPTH_M,
            azimuth=azimuth,
            dip=dip,
            sample_rate=FS,
            response=resp_usf,
        )
        channels.append(ch)

    sta = Station(
        code=sta_code,
        latitude=sta_lat,
        longitude=sta_lon,
        elevation=ELEV_M,
        creation_date=START_DATE,
        site=Site(name=sta_code),
        channels=channels,
    )
    stations_list.append(sta)

network = Network(
    code=NETWORK_CODE,
    description=NETWORK_DESCRIPTION,
    start_date=START_DATE,
    stations=stations_list,
)

inventory = Inventory(
    networks=[network],
    source="KSC Seismic Network",
)

print(inventory)

inventory.plot(projection="local", level="channel")
inventory.plot_response(min_freq=0.001, output="VEL", station="B05", channel="DHZ")

out_xml = "ksc_usf_inventory.xml"
inventory.write(out_xml, format="STATIONXML")


## Step 5 — Quick check: read back the StationXML

In [ ]:
from obspy import read_inventory

inv2 = read_inventory(out_xml)
print(inv2)

## What to submit

Submit your notebook **and** the generated StationXML file.

### Your short note (2–4 sentences)
At the end of your notebook, include a short note:
- Which stations successfully got responses from the NRL?
- Which stations did not, and why (missing mapping, missing instrument info, etc.)?

# add notes below...

Seven stations successfully got responses from the NRL: B01, B03, B05, B14, B20, B24, and B29.  
These are the stations identified in the dataframe as using the USF Nanometrics / Trillium-Centaur equipment.  
The three non-USF stations (B07, B12, and B23) were not included in the final inventory for this core exercise, because they use different Marshall equipment and the assignment said those could be ignored unless attempting bonus work.


## Full-network workflow: build StationXML for all 10 stations
- USF stations get the known Trillium Compact 120 + Centaur response
- Non-USF stations are still included in the Inventory
- For non-USF stations, keep channel-level metadata only (response=None)

In [ ]:

import pandas as pd
from obspy import UTCDateTime
from obspy.clients.nrl import NRL
from obspy.core.inventory import Inventory, Network, Station, Channel, Site

NETWORK_CODE = "1R"
NETWORK_DESCRIPTION = "KSC Seismic Network"
START_DATE = UTCDateTime(2026, 1, 7)

CHANNEL_CODES = ["DHZ", "DHN", "DHE"]
LOCATION_CODE = ""
FS = 500
ELEVATION_M = 0.0
DEPTH_M = 0.75

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def make_channel(chan_code, lat, lon, response=None):
    if chan_code.endswith("Z"):
        azimuth = 0.0
        dip = -90.0
    elif chan_code.endswith("N"):
        azimuth = 0.0
        dip = 0.0
    else:  # E
        azimuth = 90.0
        dip = 0.0

    return Channel(
        code=chan_code,
        location_code=LOCATION_CODE,
        latitude=float(lat),
        longitude=float(lon),
        elevation=ELEVATION_M,
        depth=DEPTH_M,
        azimuth=azimuth,
        dip=dip,
        sample_rate=FS,
        response=response,
    )

def is_usf_station(row):
    seismometer = safe_text(row["Seismometer"]).lower()
    digitizer = safe_text(row["Digitizer"]).lower()
    return ("trillium" in seismometer) or ("centaur" in digitizer)

# Known USF response from the NRL
nrl = NRL()

usf_sensor_keys = [
    "Nanometrics",
    "Trillium Compact 120 (Vault, Posthole, OBS)",
    "754 V/m/s",
]

usf_datalogger_keys = [
    "Nanometrics",
    "Centaur",
    "40 Vpp (1)",
    "Off",
    "Linear phase",
    str(FS),
]

resp_usf = nrl.get_response(
    sensor_keys=usf_sensor_keys,
    datalogger_keys=usf_datalogger_keys,
)

stations_list = []

for _, row in stations.iterrows():
    sta_code = safe_text(row["Site Name"])
    sta_lat = float(row["lat"])
    sta_lon = float(row["lon"])

    if is_usf_station(row):
        response_to_use = resp_usf
        response_label = "USF response"
    else:
        response_to_use = None
        response_label = "channel-only (non-USF station)"

    channels = []
    for chan_code in CHANNEL_CODES:
        ch = make_channel(chan_code, sta_lat, sta_lon, response=response_to_use)
        channels.append(ch)

    sta = Station(
        code=sta_code,
        latitude=sta_lat,
        longitude=sta_lon,
        elevation=ELEVATION_M,
        creation_date=START_DATE,
        site=Site(name=sta_code),
        channels=channels,
    )

    stations_list.append(sta)
    print(f"{sta_code}: {response_label}")

network = Network(
    code=NETWORK_CODE,
    description=NETWORK_DESCRIPTION,
    start_date=START_DATE,
    stations=stations_list,
)

inventory = Inventory(
    networks=[network],
    source="KSC Seismic Network",
)

print()
print(inventory)
print()
print("Number of stations in inventory:", len(inventory.networks[0].stations))
print("Station codes:", [sta.code for sta in inventory.networks[0].stations])

inventory.plot(projection="local", level="channel")
inventory.plot_response(min_freq=0.001, output="VEL", station="B05", channel="DHZ")

out_xml = "ksc_all10_inventory.xml"
inventory.write(out_xml, format="STATIONXML")
print(f"\nWrote StationXML: {out_xml}")
